# MediTrack: Relational Database & SQL Queries
### Project CP-02: Readmission Risk Prediction and Clinical Decision Support
This notebook demonstrates the relational database architecture and executes the three mandatory analytical queries.

In [1]:

import sqlite3
import pandas as pd

conn = sqlite3.connect('data/meditrack.db')

# Verify Database Tables & Record Counts
tables = ['patients', 'encounters', 'diagnoses', 'medications']
for t in tables:
    count = pd.read_sql_query(f"SELECT COUNT(*) FROM {t}", conn).iloc[0, 0]
    print(f"Table '{t}': {count:,} records")


Table 'patients': 69,987 records
Table 'encounters': 99,340 records
Table 'diagnoses': 298,020 records
Table 'medications': 117,953 records


In [2]:

# QUERY 1: Readmission Rate by Demographic & Prior Inpatient Cohort
q1 = '''
SELECT 
    p.age_group,
    p.gender,
    CASE 
        WHEN e.number_inpatient = 0 THEN '0 Prior Inpatient Visits'
        WHEN e.number_inpatient = 1 THEN '1 Prior Inpatient Visit'
        WHEN e.number_inpatient = 2 THEN '2 Prior Inpatient Visits'
        ELSE '3+ Prior Inpatient Visits'
    END AS inpatient_utilization_tier,
    COUNT(e.encounter_id) AS total_encounters,
    SUM(e.readmitted_30d) AS readmitted_30d_count,
    ROUND(100.0 * SUM(e.readmitted_30d) / COUNT(e.encounter_id), 2) AS readmission_rate_pct,
    ROUND(AVG(e.time_in_hospital), 2) AS avg_length_of_stay_days
FROM encounters e
JOIN patients p ON e.patient_nbr = p.patient_nbr
GROUP BY p.age_group, p.gender, inpatient_utilization_tier
HAVING COUNT(e.encounter_id) >= 50
ORDER BY readmission_rate_pct DESC
LIMIT 8;
'''
pd.read_sql_query(q1, conn)


,age_group,gender,inpatient_utilization_tier,total_encounters,readmitted_30d_count,readmission_rate_pct,avg_length_of_stay_days
0,[20-30),Female,3+ Prior Inpatient Visits,183,82,44.81,3.52
1,[30-40),Female,3+ Prior Inpatient Visits,216,80,37.04,4.62
2,[20-30),Male,3+ Prior Inpatient Visits,57,18,31.58,3.67
3,[20-30),Female,2 Prior Inpatient Visits,71,21,29.58,4.23
4,[40-50),Male,3+ Prior Inpatient Visits,377,110,29.18,4.67
5,[40-50),Female,3+ Prior Inpatient Visits,448,129,28.79,5.33
6,[50-60),Female,3+ Prior Inpatient Visits,608,174,28.62,5.00
7,[60-70),Male,3+ Prior Inpatient Visits,713,196,27.49,4.92


In [3]:

# QUERY 2: Average Length of Stay & Readmission Rate by Primary Diagnosis Group
q2 = '''
SELECT 
    d.clinical_category AS primary_diagnosis_group,
    COUNT(DISTINCT e.encounter_id) AS encounter_count,
    ROUND(AVG(e.time_in_hospital), 2) AS avg_stay_days,
    ROUND(MIN(e.time_in_hospital), 1) AS min_stay_days,
    ROUND(MAX(e.time_in_hospital), 1) AS max_stay_days,
    SUM(e.readmitted_30d) AS readmissions_30d_count,
    ROUND(100.0 * SUM(e.readmitted_30d) / COUNT(e.encounter_id), 2) AS readmission_rate_pct
FROM diagnoses d
JOIN encounters e ON d.encounter_id = e.encounter_id
WHERE d.diagnosis_seq = 1
GROUP BY d.clinical_category
ORDER BY avg_stay_days DESC;
'''
pd.read_sql_query(q2, conn)


,primary_diagnosis_group,encounter_count,avg_stay_days,min_stay_days,max_stay_days,readmissions_30d_count,readmission_rate_pct
0,Neoplasms,3131,5.28,1.0,14.0,341,10.89
1,Other,17793,4.77,1.0,14.0,2079,11.68
2,Injury,6851,4.62,1.0,14.0,850,12.41
3,Digestive,9333,4.36,1.0,14.0,1010,10.82
4,Diabetes,8661,4.33,1.0,14.0,1135,13.10
5,Genitourinary,5002,4.22,1.0,14.0,552,11.04
6,Circulatory,29680,4.21,1.0,14.0,3469,11.69
7,Respiratory,13934,4.19,1.0,14.0,1402,10.06
8,Musculoskeletal,4935,3.91,1.0,14.0,471,9.54
9,Missing_or_Other,20,3.75,1.0,8.0,5,25.00


In [4]:

# QUERY 3: Patient Encounter Sequences Tracing Readmission Trajectory (Window Functions)
q3 = '''
WITH patient_journey AS (
    SELECT 
        e.patient_nbr,
        p.age_group,
        p.gender,
        e.encounter_id,
        e.time_in_hospital AS stay_duration_days,
        e.admission_type_name,
        e.readmitted_30d,
        e.readmitted_raw,
        d.clinical_category AS primary_diagnosis,
        ROW_NUMBER() OVER (PARTITION BY e.patient_nbr ORDER BY e.encounter_id ASC) AS encounter_sequence_num,
        COUNT(e.encounter_id) OVER (PARTITION BY e.patient_nbr) AS total_patient_admissions,
        LAG(e.encounter_id) OVER (PARTITION BY e.patient_nbr ORDER BY e.encounter_id ASC) AS prior_encounter_id,
        LAG(e.time_in_hospital) OVER (PARTITION BY e.patient_nbr ORDER BY e.encounter_id ASC) AS prior_stay_duration_days
    FROM encounters e
    JOIN patients p ON e.patient_nbr = p.patient_nbr
    LEFT JOIN diagnoses d ON e.encounter_id = d.encounter_id AND d.diagnosis_seq = 1
)
SELECT 
    patient_nbr,
    encounter_sequence_num,
    total_patient_admissions,
    encounter_id,
    admission_type_name,
    primary_diagnosis,
    stay_duration_days,
    prior_encounter_id,
    prior_stay_duration_days,
    readmitted_raw,
    readmitted_30d
FROM patient_journey
WHERE total_patient_admissions > 1
ORDER BY patient_nbr, encounter_sequence_num
LIMIT 8;
'''
pd.read_sql_query(q3, conn)


,patient_nbr,encounter_sequence_num,total_patient_admissions,encounter_id,admission_type_name,primary_diagnosis,stay_duration_days,prior_encounter_id,prior_stay_duration_days,readmitted_raw,readmitted_30d
0,135,1,2,24437208,Urgent,Circulatory,8,NaN,NaN,<30,1
1,135,2,2,26264286,Emergency,Injury,3,24437208.0,8.0,>30,0
2,1152,1,5,8380170,Emergency,Other,6,NaN,NaN,>30,0
3,1152,2,5,30180318,Emergency,Other,6,8380170.0,6.0,>30,0
4,1152,3,5,55533660,Emergency,Other,10,30180318.0,6.0,>30,0
5,1152,4,5,80742510,Emergency,Other,8,55533660.0,10.0,>30,0
6,1152,5,5,83281464,Emergency,Other,12,80742510.0,8.0,NO,0
7,1314,1,3,60254142,Urgent,Injury,2,NaN,NaN,>30,0
